# Create a PyTorch3D wheel — environment check

Building PyTorch3D from source is the **hardest part** of the setup (CUDA toolkit,
matching torch, `nvcc`, C++ build tools). So build the wheel **first**: if this notebook
succeeds, your environment is ready for the full pipeline install. If it fails, the error
here tells you exactly what to fix — without running the whole install.

Runs anywhere with an NVIDIA GPU — **Colab, WSL2, or a Linux box**. The wheel is saved to
Google Drive on Colab, otherwise to a local folder you can find and copy.

A PyTorch3D wheel is only valid where **python (cp), torch, and CUDA all match** the
machine it was built on — so build it on the machine you'll run the pipeline on.

**WSL2 note:** you need the NVIDIA driver installed on **Windows** (not inside WSL) plus
the **CUDA toolkit installed inside WSL** (so `nvcc` exists), and a `torch` build matching
that CUDA. The preflight cell below checks all of this.

> ⚠️ Keep the session active during the build — the wheel is only written at the very end.

## 1. Install cvenv (from this repo)
Installs the copy of cvenv you were given. Falls back to GitHub if the repo isn't found.

In [ ]:
import sys, subprocess, pathlib

# Prefer installing cvenv from THIS repo (the folder you were sent); else GitHub.
here = pathlib.Path.cwd()
repo = next((p for p in [here, *here.parents]
             if (p / 'pyproject.toml').exists() and (p / 'cvenv').is_dir()), None)
if repo is not None:
    print('Installing cvenv from local repo:', repo)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(repo)], check=True)
else:
    print('Local repo not found from cwd; installing cvenv from GitHub.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'git+https://github.com/ribeiro-computer-vision/cvenv@v0.1.7'], check=True)

import cvenv
print('cvenv', cvenv.__version__, '| platform:', cvenv.PlatformManager().platform)

## 2. Preflight: check your environment
The wheel is compiled against your installed **torch** and **CUDA toolkit**. If `torch`
is missing, no GPU is visible, or `nvcc` isn't found, fix that first — the build can't
succeed otherwise.

In [ ]:
import os, shutil, subprocess

try:
    import torch
    print(f'\u2705 torch {torch.__version__} | CUDA {torch.version.cuda} | GPU available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        maj, mn = torch.cuda.get_device_capability(0)
        print(f'   GPU: {torch.cuda.get_device_name(0)}  (compute {maj}.{mn})')
    else:
        print('   \u26a0\ufe0f No GPU visible. A CUDA build needs a visible GPU. On WSL2 check the Windows NVIDIA driver.')
except Exception as e:
    print('\u274c torch is not importable. Install a torch build matching your CUDA first, then re-run.\n   ', e)

nvcc = shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
if os.path.exists(nvcc):
    print('\nnvcc:', nvcc)
    subprocess.run([nvcc, '--version'])
else:
    print('\n\u274c nvcc not found. Install the CUDA toolkit (matching torch\'s CUDA) so nvcc is at '
          '/usr/local/cuda or on PATH.')

## 3. Choose where to save the wheel
**Colab** → Google Drive (survives runtime resets). **Otherwise** (WSL / local / server)
→ a local `cvenv_wheels/` folder in the current directory, easy to find and copy.

In [ ]:
import os, cvenv

if cvenv.PlatformManager().platform == 'Colab':
    from google.colab import drive
    drive.mount('/content/drive')
    WHEEL_DIR = '/content/drive/MyDrive/cvenv_wheels'
else:
    WHEEL_DIR = os.path.abspath('cvenv_wheels')

os.makedirs(WHEEL_DIR, exist_ok=True)
print('Wheel will be saved to:', WHEEL_DIR)

## 4. Build the wheel
Compiles PyTorch3D from source with all the fixes cvenv applies automatically: builds for
**your GPU's arch only**, sets the **CUDA-13 pulsar flag** (`-static-global-template-stub=false`),
and uses the **system linker** inside conda envs. Takes ~10–30 min.

The build is **idempotent**: if a wheel is already in `WHEEL_DIR` it is reused and nothing
is recompiled. Set `FORCE_REBUILD = True` below to rebuild instead — do that when torch,
CUDA or python has changed on this machine, since a wheel is only valid where all three
match.

### Rebuild or reuse?
The cell below reports any wheel already in `WHEEL_DIR` and what it was built for, then
applies your choice. Reuse takes seconds; a rebuild takes ~10–30 min.

> A wheel filename records the **python** tag and platform, but **not** torch or CUDA. So a
> matching `cp` tag is necessary, not sufficient — on Colab especially, the runtime's torch
> can change under a wheel left in Drive. If `verify()` later fails with an `_C`
> `undefined symbol` error, that is the signal to come back and rebuild.

In [ ]:
# ----------------------------------------------------------------------
# False -> reuse a wheel already in WHEEL_DIR (fast; the usual case)
# True  -> recompile from source; the fresh wheel becomes the one used
FORCE_REBUILD = False
# ----------------------------------------------------------------------

import glob, os, sys, time

this_py = f"cp{sys.version_info.major}{sys.version_info.minor}"
found = sorted(glob.glob(os.path.join(WHEEL_DIR, "pytorch3d-*.whl")),
               key=os.path.getmtime, reverse=True)

if not found:
    print(f"No pytorch3d wheel in {WHEEL_DIR}")
    print("The next cell will build one (~10-30 min), whatever FORCE_REBUILD says.")
else:
    print(f"{len(found)} wheel(s) already in {WHEEL_DIR}:\n")
    for w in found:
        st, name = os.stat(w), os.path.basename(w)
        tag = next((p for p in name.split("-") if p.startswith("cp")), "?")
        built = time.strftime("%Y-%m-%d %H:%M", time.localtime(st.st_mtime))
        note = "matches this runtime" if tag == this_py else f"BUILT FOR {tag}, you are on {this_py}"
        print(f"  {name}")
        print(f"     {st.st_size / 1e6:.1f} MB   built {built}   [{note}]")

    newest = os.path.basename(found[0])
    newest_tag = next((p for p in newest.split("-") if p.startswith("cp")), "?")
    print()
    if FORCE_REBUILD:
        print(f"FORCE_REBUILD = True  -> rebuilding; the new wheel supersedes {newest}")
    elif newest_tag != this_py:
        print(f"!! {newest}")
        print(f"   was built for {newest_tag} but this runtime is {this_py}, so installing")
        print("   it will fail. Set FORCE_REBUILD = True above and re-run this cell.")
    else:
        print(f"FORCE_REBUILD = False -> reusing {newest} (no compile)")
        print("   Set FORCE_REBUILD = True above to rebuild for this runtime.")

In [ ]:
whl = cvenv.get_component('pytorch3d').build_wheel(out_dir=WHEEL_DIR, force=FORCE_REBUILD)
print('\n✅ wheel to install:', whl)

## 5. Install & verify
`verify()` runs `import pytorch3d._C` — the real proof the compiled extension matches this
machine's torch/CUDA. If you see `✅ pytorch3d … (_C OK)`, **your environment is good to go.**

In [ ]:
cvenv.get_component('pytorch3d').install(wheel_url=whl)
cvenv.get_component('pytorch3d').verify()

## Done — and how to reuse it
That `.whl` in `WHEEL_DIR` is a normal file. On the **same** machine/stack (same python,
torch, CUDA), skip the build next time — point `wheel_url` at it:

```python
import cvenv, glob, os
whl = max(glob.glob(os.path.join(WHEEL_DIR, 'pytorch3d-*.whl')), key=os.path.getmtime)
cvenv.get_component('pytorch3d').install(wheel_url=whl)
```

If torch or CUDA later changes on this machine, `verify()` will fail with an `_C`
`undefined symbol` error — just re-run this notebook with `FORCE_REBUILD = True` to rebuild.

**If the build failed:** the message tells you what your environment is missing (no GPU,
no `nvcc`, torch/CUDA mismatch). Fix that and re-run — you don't need to attempt the full
pipeline install until this wheel builds and `verify()` passes.